# Our extractor — redaction check

Cohort = same 14 docs as [nb_ds_esn_identifier_smoke_test.ipynb](nb_ds_esn_identifier_smoke_test.ipynb).

Per Pranesh (2026-05-06): our pipeline `nb_sdg_fsr_metadata.py` is a direct port of the DS team's FSR Scraping POC (`normalize_json_llm_comb.py`). We already have our pipeline's output in `vaid…biz_metadata_field_service_report.esn`. The remaining question is narrow:

> For the 4 manual ecrt docs where our pipeline stored `XXXXXX`/`XXXXXXX`, what does the real PDF text contain, and would our existing page-1 regex (`_extract_esn_from_page1_text`) recover a real ESN if we taught the resolver to treat `^X{4,}$` as null?

**This notebook does not call the LLM.** It just (a) reads page-1 text via PyMuPDF, (b) runs our existing regex, (c) prints results next to the stored ESN and the DS LLM top-1 so we can decide if a tiny resolver patch is enough.

Cells: install → config → page-1 reader → our regex → comparison table.

## 0. Install dependencies

In [ ]:
%pip install pymupdf --quiet
dbutils.library.restartPython()

## 1. Config — same cohort + same dev metadata table

In [ ]:
METADATA_TABLE = "vaid.ai_sot_field_service_report.biz_metadata_field_service_report"

DOC_IDS = [
    "d3b1da8a-03b4-4ea6-aa7b-0482edb532ce",
    "3ef8d150-1225-4c62-ac5e-ae9166c73954",
    "913433ad-6d19-472f-917c-5048f689a5ed",
    "c020379c-93a4-49f6-b6a9-1388edd59d3a",
    "03d3bef2-db2b-4311-b147-641025e3636b",
    "36de1c13-acee-4633-8b76-8a262f738a20",
    "34bc62b0-573c-4f43-bc62-b0573ccf43e9",
    "348ddaa2-8c32-4646-9f17-c57f33256d82",
    "4b70aab1-e210-4d5b-9cbf-bfe690436861",
    "d5334dce-9d7f-4fa1-b34d-ce9d7f7fa130",
    "090dbba1800f4f5b",
    "090dbba18003a23c",
    "090dbba180100488",
    "090dbba1800b5e28",
]

# DS extractor top-1 results from the prior smoke test (paste from CSV — used for side-by-side display only).
DS_TOP1 = {
    "d3b1da8a-03b4-4ea6-aa7b-0482edb532ce": "336X233",
    "3ef8d150-1225-4c62-ac5e-ae9166c73954": "297843",
    "913433ad-6d19-472f-917c-5048f689a5ed": "270T434",
    "c020379c-93a4-49f6-b6a9-1388edd59d3a": "270T577",
    "03d3bef2-db2b-4311-b147-641025e3636b": "270T658",
    "36de1c13-acee-4633-8b76-8a262f738a20": "297905",
    "34bc62b0-573c-4f43-bc62-b0573ccf43e9": None,        # error in DS run
    "348ddaa2-8c32-4646-9f17-c57f33256d82": "297898",
    "4b70aab1-e210-4d5b-9cbf-bfe690436861": "298374",
    "d5334dce-9d7f-4fa1-b34d-ce9d7f7fa130": "297897",
    "090dbba1800f4f5b":                     "298126",
    "090dbba18003a23c":                     "0298304",
    "090dbba180100488":                     "0298304",
    "090dbba1800b5e28":                     "297897",
}

print(f"Cohort size: {len(DOC_IDS)}")

## 2. Resolve cohort → volume_path + current_stored_esn

In [ ]:
from pyspark.sql import functions as F

meta_df = (
    spark.table(METADATA_TABLE)
    .filter(F.col("document_id").isin(DOC_IDS))
    .select("document_id", "volume_path", "esn", "esn_source")
    .toPandas()
    .set_index("document_id")
)
print(f"Resolved {len(meta_df)} of {len(DOC_IDS)} docs")
display(spark.createDataFrame(meta_df.reset_index()))

## 3. Page-1 text reader + our existing regex (lifted verbatim from `nb_sdg_fsr_metadata.py`)

In [ ]:
import re, fitz

def read_page1_text(volume_path: str) -> str:
    """Read first page text via PyMuPDF — same approach as our pipeline's page1 extraction."""
    with fitz.open(volume_path) as doc:
        if doc.page_count == 0:
            return ""
        return doc.load_page(0).get_text("text") or ""

# Verbatim from pw_sdg_ai_ser_repo/silver/src/etl/nb_sdg_fsr_metadata.py L449-462
def _extract_esn_from_page1_text(text: str) -> str:
    if not text:
        return ""
    cleaned = re.sub(r"\s+", " ", text)
    patterns = [
        r"(?i)\bESN(?:\s*/\s*SY)?\s*[:#-]?\s*([A-Z0-9-]{4,})",
        r"(?i)\bEQUIP(?:MENT)?\s+SERIAL\s+(?:NO|NUMBER)?\s*[:#-]?\s*([A-Z0-9-]{4,})",
    ]
    for pat in patterns:
        m = re.search(pat, cleaned)
        if m:
            return m.group(1).strip()
    return ""

# Helper: would the proposed redaction filter strip this value?
_REDACTION_RE = re.compile(r"^X{4,}$", re.IGNORECASE)
def is_redacted(value: str) -> bool:
    return bool(value) and bool(_REDACTION_RE.match(value.strip()))

print("helpers ready")

## 4. Run regex on page-1 text for each doc

In [ ]:
rows = []
for doc_id in DOC_IDS:
    if doc_id not in meta_df.index:
        rows.append({"document_id": doc_id, "error": "not in dev metadata"})
        continue
    vp = meta_df.loc[doc_id, "volume_path"]
    stored = meta_df.loc[doc_id, "esn"]
    try:
        page1 = read_page1_text(vp)
        page1_regex_esn = _extract_esn_from_page1_text(page1)
    except Exception as e:
        rows.append({"document_id": doc_id, "error": f"{type(e).__name__}: {e}"})
        continue
    rows.append({
        "document_id":     doc_id,
        "volume_path":     vp,
        "stored_esn":      stored,
        "stored_redacted": is_redacted(stored),
        "page1_chars":     len(page1),
        "page1_regex_esn": page1_regex_esn,
        "ds_top1_esn":     DS_TOP1.get(doc_id),
        "page1_text":      page1,
        "error":           None,
    })
    print(f"{doc_id[:40]:40s}  stored={stored!r:20s}  redacted={is_redacted(stored)!s:5s}  page1_regex={page1_regex_esn!r:20s}  ds_top1={DS_TOP1.get(doc_id)!r}")

## 5. Comparison table

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)

# Apply the proposed resolver patch: if stored is redacted, prefer page-1 regex; else keep stored.
df["resolved_with_patch"] = df.apply(
    lambda r: r["page1_regex_esn"] if r.get("stored_redacted") and r.get("page1_regex_esn") else r.get("stored_esn"),
    axis=1,
)
df["patch_matches_ds"] = df["resolved_with_patch"] == df["ds_top1_esn"]
df["patch_changed_value"] = df["resolved_with_patch"] != df["stored_esn"]

cols = ["document_id", "stored_esn", "stored_redacted", "page1_chars", "page1_regex_esn", "ds_top1_esn", "resolved_with_patch", "patch_changed_value", "patch_matches_ds"]
comparison_df = df[cols].copy()
display(spark.createDataFrame(comparison_df))  # downloadable from Databricks UI

## 6. Page-1 text snippets for the redacted docs (manual spot-check)

Look here to confirm whether real ESNs are visible in the PDF text. If yes → DS LLM is recovering them, our regex may or may not be.

In [ ]:
redacted_rows = df[df["stored_redacted"] == True]
for _, r in redacted_rows.iterrows():
    print("=" * 80)
    print(f"doc_id:           {r['document_id']}")
    print(f"stored:           {r['stored_esn']!r}")
    print(f"page1 regex hit:  {r['page1_regex_esn']!r}")
    print(f"DS LLM top-1:     {r['ds_top1_esn']!r}")
    print(f"page1 text ({r['page1_chars']} chars):")
    print("-" * 80)
    print(r["page1_text"][:3000])
    print()

## 7. Summary

In [ ]:
ok = df[df["error"].isna()]
n = len(ok)
n_redacted = ok["stored_redacted"].sum()
n_redacted_recovered = ((ok["stored_redacted"]) & (ok["page1_regex_esn"].astype(bool))).sum()
n_patch_matches_ds = ok["patch_matches_ds"].sum()
n_patch_changed = ok["patch_changed_value"].sum()

print(f"Docs analyzed:                                 {n}")
print(f"  stored value is redacted (^X{{4,}}$):         {n_redacted}")
print(f"  redacted docs where page-1 regex finds ESN:  {n_redacted_recovered}")
print(f"  patch changes value vs stored:               {n_patch_changed}")
print(f"  patch result matches DS LLM top-1:           {n_patch_matches_ds}")
print()
print("Read this as:")
print("  If 'page1 regex finds ESN' == 'redacted count'  -> tiny resolver patch fixes redaction problem, no DS lift needed.")
print("  If less                                          -> page1 regex misses some; need DS LLM (or improve regex / use full-doc text).")